# Sentiment EDA

Phase 2 notebook for sentiment class balance, review length, star ratings, and top TF-IDF terms.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_PATH = Path('../data/raw/reviews.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/raw/reviews.csv')

reviews = pd.read_csv(DATA_PATH)
reviews.head()

In [ ]:
print('Shape:', reviews.shape)
display(reviews.info())
display(reviews.isna().sum())
display(reviews['sentiment_label'].value_counts())

In [ ]:
label_counts = reviews['sentiment_label'].value_counts().sort_index()
label_counts.plot(kind='bar', title='Sentiment Class Balance')
plt.xlabel('sentiment_label')
plt.ylabel('reviews')
plt.show()

In [ ]:
reviews['review_length_words'] = reviews['review_text'].str.split().str.len()
display(reviews.groupby('sentiment_label')['review_length_words'].agg(['count', 'mean', 'median', 'min', 'max']).round(2))
reviews.boxplot(column='review_length_words', by='sentiment_label')
plt.title('Review Length by Sentiment')
plt.suptitle('')
plt.xlabel('sentiment_label')
plt.ylabel('word count')
plt.show()

In [ ]:
display(reviews.groupby('sentiment_label')['star_rating'].agg(['mean', 'min', 'max']).round(2))

In [ ]:
top_terms = {}
for label in sorted(reviews['sentiment_label'].unique()):
    texts = reviews.loc[reviews['sentiment_label'] == label, 'review_text']
    vectorizer = TfidfVectorizer(stop_words='english', max_features=20, ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(texts)
    scores = matrix.mean(axis=0).A1
    terms = sorted(zip(vectorizer.get_feature_names_out(), scores), key=lambda item: item[1], reverse=True)[:10]
    top_terms[label] = [term for term, score in terms]

display(pd.DataFrame.from_dict(top_terms, orient='index').T)